[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/50_gae_solution.ipynb)

#  Solution: GAE Advantage Estimation

Reference: Schulman et al. "High-Dimensional Continuous Control Using Generalized Advantage Estimation" (ICLR 2016). Used in PPO, GRPO, and most modern RL policies for LLM agents.

```text
Backward loop O(T):
  A_last = 0
  for t from T-1 down to 0:
    delta = r_t + gamma * V_{t+1} * (1-done) - V_t
    A_t = delta + gamma * lam * (1-done) * A_{t+1}
  Return advantages, returns (A + V)
```

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
#  SOLUTION

def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    """Compute Generalized Advantage Estimation.

    Reference: Schulman et al. ICLR 2016.
    GAE(lam) = (1-lam)*(TD(0) + lam*TD(1) + lam^2*TD(2) + ...)
    Key insight: backward loop computes this exponentially-weighted sum in O(T).
    Used in: PPO, TRPO, GRPO (DeepSeek-R1), and RLHF training pipelines.
    """
    T = len(rewards)
    advantages = torch.zeros(T)
    last_adv = 0.0

    for t in reversed(range(T)):
        mask = 0.0 if dones[t] else 1.0
        delta = rewards[t] + gamma * values[t+1] * mask - values[t]
        last_adv = delta + gamma * lam * mask * last_adv
        advantages[t] = last_adv

    returns = advantages + values[:T]
    return advantages, returns

In [ ]:
#  Verify
r = torch.tensor([1.0, 0.0, 2.0])
v = torch.tensor([0.0, 0.5, 0.5, 0.0])
d = torch.tensor([False, True, False])
adv, ret = compute_gae(r, v, d, gamma=0.9, lam=0.5)
print(f"Advantages: {adv}")
print(f"Returns:    {ret}")

In [ ]:
from torch_judge import check
check("gae")